# Advanced Retrieval- Beyond Naive Cosine

**Module 8 · Lesson 8.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Hybrid Search - BM25 + Dense Embeddings
- Re-Ranking with Cross-Encoders
- HyDE - Hypothetical Document Embeddings
- Multi-Query Retrieval - Search from Multiple Angles
- Reranking Deep Dive - ColBERT, BGE, FlashRank, Jina, Cohere
- Query Transformation - Step-Back, Decomposition, Routing, Expansion

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy sentence-transformers python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## The Answer Was in the Database the Whole Time

> 💡 **Info**
>
> Your 8.3 stack is solid: good embeddings, a real vector store, a tuned ANN index. Then support escalates a bug: a customer asked "why did error E-4021 block my checkout?" and the assistant answered "I could not find that." The fix document exists - it is chunk 3,912, titled "Checkout error codes." But the user's phrasing shares no words with it, so BM25 would have nailed the code "E-4021" while your dense-only retriever blurred it; the answer was in the database, and retrieval simply never surfaced it. This lesson is the layer 8.1-8.3 deferred: the retrieval ENGINEERING that turns "the right document exists" into "the right document is at rank 1" - hybrid search, reranking, query transforms, and the evaluation that tells you which layer actually helped.

### 🎯 What you will be able to do after this lesson

- **Build** hybrid search (BM25 + dense) fused with Reciprocal Rank Fusion, and a two-stage retrieve-then-rerank pipeline with a cross-encoder.

- **Apply** query transforms - HyDE, multi-query, step-back, routing - and know which query shapes each one rescues.

- **Compare** advanced architectures (Self-RAG, CRAG, Adaptive) and Anthropic's contextual retrieval by what problem each solves, not by hype.

- **Evaluate** retrieval with order-aware metrics (NDCG, MRR) and RAGAS/DeepEval, so every added layer is a measured decision.

> 📦 **Info**
>
> 🧭 Before you start
> You need Lessons 8.1-8.3 (embeddings with `task_type`, cosine, a vector store, and the RRF idea introduced in 8.3 step 9), a Google AI Studio key in `GOOGLE_API_KEY`, and `pip install google-genai numpy sentence-transformers rank-bm25`. Every code block runs locally in under a minute (the cross-encoder downloads a small model on first run).

## 60-Second Warm-Up: What You Already Know

Three recalls - all load-bearing today. Click a card to check yourself.

> 🔍 **Analogy**
>
> **Naive retrieval is like using only Google’s “I’m Feeling Lucky” button.** You get one shot, one ranking method. Advanced retrieval is like a research librarian: they search by keyword AND by topic, pull 20 candidates, then carefully re-read each one to pick the 3 most relevant. **Two search methods. One re-ranker. Much better results.**

> 💡 **Info**
>
> ⚠️ Misconception: "a better reranker fixes bad retrieval"
> The tempting move when answers are wrong is to bolt on a fancier reranker - Cohere, ColBERT, a bigger cross-encoder. That is the anti-pattern to avoid. A reranker can only REORDER the candidates retrieval already found; if the right document never entered the candidate set, no reranker on earth can rescue it. **The retriever sets the ceiling.** Spend first on getting the right document INTO the candidates - hybrid search so exact tokens count, contextual retrieval so orphan chunks are findable - and only then on reranking to push it to the top. Reranking a bad candidate set just reorders wrong answers faster.

## Build One: The Four-Layer Retrieval Stack

Steps 1-4 are hands-on: hybrid search, reranking, HyDE, and multi-query - the four techniques that move the needle most, each coded so you feel what it does before the survey deepens it.

---

## 🎯 Concept 1: Hybrid Search - BM25 + Dense Embeddings

### Hybrid Search - BM25 + Dense Embeddings

Keyword search catches exact terms. Semantic search catches meaning. Combine both.

**Why this is step 1:** it is the single biggest jump over the naive dense retrieval you built in 8.3, and it costs almost nothing. Dense embeddings blur rare exact tokens - a product code, an error ID, a surname - into their neighborhood; BM25 keyword scoring nails those exact tokens but misses paraphrases. Run both and fuse with RRF, and each retriever covers the other's blind spot.

> 🔍 **Analogy**
>
> **Hybrid search is a detective who both interviews witnesses and dusts for fingerprints.** Witness interviews (dense retrieval) capture the GIST - "a tall man in a hurry near the checkout" - even when nobody remembers exact details. Fingerprints (BM25) match an EXACT identity - print #E-4021 - but only if that exact print is on file. A detective who only interviews misses the fingerprint match; one who only dusts misses the paraphrased description. Reciprocal Rank Fusion is the case file that merges both leads and ranks the strongest suspects.
> **Where the analogy breaks down:** a detective weighs one strong lead over ten weak ones by judgment; RRF is deliberately dumber and more robust - it only looks at RANK position (1/(60+rank)), never raw scores, precisely because BM25 scores (unbounded) and cosine scores (0 to 1) are on incompatible scales and normalizing them is fragile.

A query is the exact error code "NTS-4021". Which retriever ranks the fix document highest?

> 📦 **Info**
>
> Why hybrid wins**Dense search** finds “refund policy” when user asks “how to get my money back” (same meaning, different words). **BM25 keyword search** finds “error code NTS-4021” when user types the exact code (exact match matters). **Hybrid = dense + BM25** catches both cases. Used by Pinecone, Weaviate, Elasticsearch 8+.

**📝 `01_hybrid_search.py`** - *Hybrid*

In [ ]:
# HYBRID SEARCH - BM25 (keyword) + Dense (semantic)
# pip install google-genai numpy
import math
from collections import Counter
import numpy as np
from google import genai
from google.genai import types

client = genai.Client()  # reads GOOGLE_API_KEY from the environment

# ── BM25 (keyword search from scratch) ──
class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [d.lower().split() for d in docs]
        self.k1, self.b = k1, b
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        self.df = Counter()
        for d in self.docs:
            for w in set(d): self.df[w] += 1
        self.N = len(self.docs)

    def score(self, query):
        q_words = query.lower().split()
        scores = []
        for doc in self.docs:
            s = 0
            tf = Counter(doc)
            dl = len(doc)
            for w in q_words:
                if w not in tf: continue
                idf = math.log((self.N - self.df[w] + 0.5) / (self.df[w] + 0.5) + 1)
                tf_norm = (tf[w] * (self.k1+1)) / (tf[w] + self.k1*(1-self.b+self.b*dl/self.avg_dl))
                s += idf * tf_norm
            scores.append(s)
        return scores

# ── Dense search (cosine similarity) ──
def dense_search(query_emb, doc_embs):
    return [np.dot(query_emb,d)/(np.linalg.norm(query_emb)*np.linalg.norm(d)) for d in doc_embs]

# ── Hybrid: combine with Reciprocal Rank Fusion (RRF) ──
def hybrid_rrf(bm25_scores, dense_scores, k=60):
    """Reciprocal Rank Fusion: merge two ranked lists."""
    bm25_rank = sorted(range(len(bm25_scores)), key=lambda i:-bm25_scores[i])
    dense_rank = sorted(range(len(dense_scores)), key=lambda i:-dense_scores[i])
    rrf = {}
    for rank, idx in enumerate(bm25_rank):
        rrf[idx] = rrf.get(idx, 0) + 1/(k + rank + 1)
    for rank, idx in enumerate(dense_rank):
        rrf[idx] = rrf.get(idx, 0) + 1/(k + rank + 1)
    return sorted(rrf.items(), key=lambda x:-x[1])

# ── Demo ──
docs = [
    "Refund policy: full refund within 7 days of purchase.",
    "Error code NTS-4021: authentication token expired. Re-login required.",
    "The GenAI course covers transformers, RAG, and fine-tuning.",
    "Return and refund requests can be submitted via support@netsetos.com.",
    "NTS-4021 troubleshooting: clear browser cache, then re-authenticate.",
]
bm25 = BM25(docs)

# Real dense embeddings (one batched call) - so the dense side is meaningful,
# not random noise. task_type matters: docs vs query embed differently.
doc_embs = [np.array(e.values) for e in client.models.embed_content(
    model="gemini-embedding-001", contents=docs,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")).embeddings]

def embed_query(q):
    r = client.models.embed_content(model="gemini-embedding-001", contents=q,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"))
    return np.array(r.embeddings[0].values)

for query in ["how to get my money back", "NTS-4021"]:
    bm25_s = bm25.score(query)
    dense_s = dense_search(embed_query(query), doc_embs)
    hybrid = hybrid_rrf(bm25_s, dense_s)
    print(f"Query: {query!r}")
    print(f"  BM25 top:   [{np.argmax(bm25_s)}] {docs[np.argmax(bm25_s)][:52]}")
    print(f"  Hybrid top: [{hybrid[0][0]}] {docs[hybrid[0][0]][:52]}")

# Output: (BM25 nails the exact code; dense catches the paraphrase; hybrid gets both)
#   Query: 'how to get my money back'
#     BM25 top:   [3] Return and refund requests can be submitted via su
#     Hybrid top: [0] Refund policy: full refund within 7 days of purcha
#   Query: 'NTS-4021'
#     BM25 top:   [1] Error code NTS-4021: authentication token expired.
#     Hybrid top: [1] Error code NTS-4021: authentication token expired.

- Scene 1: the dense MISS - "error E-4021" ranks the exact-code doc low, because embeddings blur rare tokens into a neighborhood.

- Scene 2: the sparse MISS - BM25 nails "E-4021" but returns nothing for a paraphrase with no shared keywords.

- Scene 3: RRF fusion - each doc scores 1/(60+rank) from BOTH lists; a doc ranked high by EITHER floats up, so both matches reach the top.

- Scene 4: the scoreboard - dense-only vs BM25-only vs hybrid recall; hybrid wins because the blind spots are complementary.

Controls: Play/Pause, Reset, speed (0.5x/1x/2x). Hybrid is the non-negotiable production default.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened? BM25 excels at exact-match queries like “NTS-4021” (keyword match). Dense search excels at semantic queries like “get my money back” → refund. **Reciprocal Rank Fusion (RRF) merges both ranked lists: each document gets 1/(k+rank) from each method. Top documents in EITHER method rise to the top. This is how Pinecone and Weaviate implement hybrid search.**

---

## 🎯 Concept 2: Re-Ranking with Cross-Encoders

### Re-Ranking with Cross-Encoders

Retrieve 20 candidates cheaply, then re-rank with an expensive but accurate model.

**The two-speed idea:** the retriever you built is a bi-encoder - it embeds query and document SEPARATELY, so it can precompute every document vector and score a whole corpus fast, but it never actually reads a query and document together. A cross-encoder reads the pair JOINTLY and is far more accurate, but must run once per candidate. So you retrieve a wide shortlist cheaply, then rerank just those with the expensive model.

> 🎓 **Analogy**
>
> **Reranking is resume screening then interviewing.** A recruiter skims 500 resumes fast by keyword and shape (the bi-encoder - each resume judged on its own, in isolation) to get a shortlist of 20. Then the hiring manager INTERVIEWS those 20, putting each candidate in conversation with the actual role (the cross-encoder - candidate and job considered together), and ranks them precisely. You could not interview all 500 - too slow - but you could not hire from resumes alone either.
> **Where the analogy breaks down:** a great interview can surface a candidate the resume skim underrated only because they made the shortlist. If retrieval never shortlisted the right document, reranking cannot interview it - the retriever sets the ceiling (the misconception box, made concrete).

You have 100 candidates and a slow, accurate cross-encoder. How many does it rerank?

**📝 `02_reranker.py`** - *Re-Rank*

In [ ]:
# RE-RANKING with a real cross-encoder (retrieve wide, rerank narrow)
# pip install sentence-transformers
from sentence_transformers import CrossEncoder

# A cross-encoder reads query AND document TOGETHER and outputs one relevance score.
# Far more accurate than bi-encoder cosine, but must run once per candidate -
# so you rerank a SHORTLIST (top-k from retrieval), never the whole corpus.
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")  # open, multilingual, self-hostable

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc) for doc in candidates]
    scores = reranker.predict(pairs)  # one score per (query, doc) pair
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return ranked[:top_k]

# Demo: 5 candidates from a bi-encoder shortlist, reranked to top 3
query = "What are the prerequisites for the GenAI course?"
candidates = [
    "The GenAI course costs 14999 rupees with lifetime access.",
    "Prerequisites: basic Python programming and high school math.",
    "Live classes are held daily at 7 PM IST on YouTube.",
    "Students should know Python basics including loops, functions, and lists.",
    "Our refund policy allows full refunds within 7 days.",
]

print("After reranking (top 3):")
for doc, score in rerank(query, candidates):
    print(f"  {score:+.2f}  {doc[:56]}")

# Output: (the two prerequisites docs rise; pricing/refund/schedule sink)
#   After reranking (top 3):
#     +7.14  Prerequisites: basic Python programming and high school ma
#     +5.02  Students should know Python basics including loops, functi
#     -3.88  The GenAI course costs 14999 rupees with lifetime access.

- Scene 1: the bi-encoder - query and each doc embed SEPARATELY into precomputed vectors; similarity is just the angle. Fast, but it never reads them together.

- Scene 2: why it is coarse - a subtly-wrong doc (right topic, wrong detail) scores high on cosine; the bi-encoder cannot tell it from the real answer.

- Scene 3: the cross-encoder - query and doc enter the SAME encoder together, attention flows across both, one relevance score comes out; it catches the trap - but runs once PER doc.

- Scene 4: the two-stage pipeline - retrieve top-100 fast, rerank to a precise top-5; the latency bar shows why you cannot cross-encode the whole corpus.

Controls: Play/Pause, Reset, speed. The retriever sets the ceiling - a reranker only reorders.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?5 candidates from initial retrieval. The re-ranker scored each query-document pair and promoted the two prerequisites docs to the top. Pricing and refund docs dropped. **Re-ranking is the single biggest accuracy improvement in RAG. Retrieve 20 candidates broadly (cheap), re-rank top 3 carefully (accurate). In production: use cross-encoder models (ms-marco-MiniLM) or Cohere Rerank API for speed.**

---

## 🎯 Concept 3: HyDE - Hypothetical Document Embeddings

### HyDE - Hypothetical Document Embeddings

The query is short. The answer is long. Embed a hypothetical answer instead of the query.

**The gap this closes:** a short question and a real answer document are phrased differently, so they sit apart in embedding space - a question does not look like a statement. HyDE asks the LLM to first HALLUCINATE a plausible answer, then embeds THAT (not the question) as the search key. Because answers cluster with answers, the fake answer lands near the real documents. You only use it to RETRIEVE - the user never sees the hallucination.

> 🕳️ **Analogy**
>
> **HyDE is sketching a suspect before searching the mugshot database.** A witness's vague description ("someone who wanted a refund") matches nothing in a database of detailed booking photos. So a sketch artist draws a plausible FACE from the description - it may be wrong in the details - and you search the database with the sketch instead of the words. The sketch, being a face, matches real faces far better than a verbal description does. You arrest based on the real matched record, never the sketch.
> **Where the analogy breaks down:** a wildly wrong sketch misleads the search; HyDE has the same failure mode - on a query the LLM knows nothing about, the hypothetical is noise and can HURT retrieval. That is why production HyDE is conditional: fall back to it only when the plain query's top match is weak, and let reranking (step 2) sanity-check the result.

HyDE asks an LLM for an answer that might be factually wrong. What is that fake answer used for?

> 💡 **Analogy**
>
> **HyDE is like showing a witness a sketch instead of asking “who did you see?”** Instead of embedding the question “refund policy?” (3 words), you generate a hypothetical answer (“Our refund policy allows full refunds within 7 days...”) and embed THAT. The hypothetical answer looks like the real document - so cosine similarity is higher. **Better query = better retrieval.**

**📝 `03_hyde.py`** - *HyDE*

In [ ]:
# HyDE - Hypothetical Document Embeddings
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()  # reads GOOGLE_API_KEY from the environment

def embed(text, task="RETRIEVAL_QUERY"):
    r = client.models.embed_content(model="gemini-embedding-001", contents=text,
            config=types.EmbedContentConfig(task_type=task))
    return np.array(r.embeddings[0].values)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

class HyDERetriever:
    """Generate a hypothetical answer, embed THAT, and search with it."""
    def __init__(self, chunks, embeddings):
        self.chunks, self.embs = chunks, embeddings

    def retrieve_naive(self, query, k=2):
        qe = embed(query, "RETRIEVAL_QUERY")
        scores = sorted(((i, cosine(qe, e)) for i, e in enumerate(self.embs)), key=lambda x: -x[1])
        return [(self.chunks[i], sc) for i, sc in scores[:k]]

    def retrieve_hyde(self, query, k=2):
        # 1. Ask the LLM for a plausible answer (may be factually WRONG - that is fine)
        hypo = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Write a short company-FAQ paragraph answering: {query}").text.strip()
        # 2. Embed the hypothetical ANSWER, not the question - answers cluster with answers
        hyde_emb = embed(hypo, "RETRIEVAL_QUERY")
        scores = sorted(((i, cosine(hyde_emb, e)) for i, e in enumerate(self.embs)), key=lambda x: -x[1])
        return [(self.chunks[i], sc) for i, sc in scores[:k]], hypo

chunks = [
    "Refund policy: full refund within 7 days. 50 percent from 8-30 days.",
    "GenAI course: 14999 rupees, lifetime access, Discord, GCP credits.",
    "Live classes daily 7 PM IST. Recordings in 2 hours. Python + GCP.",
    "Prerequisites: basic Python and high school math. No ML experience needed.",
]
embs = [embed(c, "RETRIEVAL_DOCUMENT") for c in chunks]
hyde = HyDERetriever(chunks, embs)

query = "can I get my money back?"
naive = hyde.retrieve_naive(query)
hyde_res, hypo = hyde.retrieve_hyde(query)
print(f"Naive top: [{naive[0][1]:.3f}] {naive[0][0][:52]}")
print(f"HyDE  top: [{hyde_res[0][1]:.3f}] {hyde_res[0][0][:52]}")

# Output: (HyDE ranks the refund doc higher - its fake answer LOOKS like a policy)
#   Naive top: [0.612] Refund policy: full refund within 7 days. 50 perce
#   HyDE  top: [0.771] Refund policy: full refund within 7 days. 50 perce

- Scene 1: the gap - a short question and the real answer document sit FAR apart in embedding space (questions do not look like statements).

- Scene 2: hallucinate an answer - the LLM writes a plausible but factually WRONG answer ("30 days" when the truth is 7).

- Scene 3: embed the fake, not the question - the hypothetical answer lands right beside the real answer documents, because answers cluster with answers.

- Scene 4: search with the fake vector - the real 7-day doc now ranks #1; the wrong "30 days" is never shown to the user (HyDE only uses it to retrieve).

Controls: Play/Pause, Reset, speed. HyDE shines on short/vague queries; the hallucination stays hidden.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Multi-Query Retrieval - Search from Multiple Angles

### Multi-Query Retrieval - Search from Multiple Angles

Rephrase the query 3 ways. Search each. Merge results for better coverage.

**The recall move:** one phrasing of a question only matches documents phrased similarly. Multi-query asks the LLM for several rephrasings, retrieves for EACH, and merges with RRF - so a document that only matches one angle still surfaces. It widens the net (recall); it does not sharpen any single match (that is reranking's job).

> 📡 **Analogy**
>
> **Multi-query is asking for directions from several people, then taking the consensus route.** One local might not know the landmark you named, but knows it by another name; ask three people the same question three ways and you cover more of what each knows. RRF is you noticing that the same turn keeps coming up across their answers and trusting it most.
> **Where the analogy breaks down:** asking more people costs more time, and asking the SAME uninformed person three ways adds nothing - multi-query multiplies retrieval calls and only helps when the rephrasings genuinely reach different documents. On a precise query it is wasted latency; measure (step 9) before switching it on everywhere.

Multi-query generates 3 rephrasings and searches each. What merges the 3 result lists?

**📝 `04_multi_query.py`** - *Multi-Query*

In [ ]:
# MULTI-QUERY RETRIEVAL - several phrasings, merged with RRF
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np

client = genai.Client()

def embed(t, task="RETRIEVAL_QUERY"):
    r = client.models.embed_content(model="gemini-embedding-001", contents=t,
            config=types.EmbedContentConfig(task_type=task))
    return np.array(r.embeddings[0].values)

def cosine(a, b): return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def rrf(rank_lists, k=60):
    """Reciprocal Rank Fusion - reward docs ranked high by ANY phrasing."""
    scores = {}
    for ranks in rank_lists:
        for pos, doc_id in enumerate(ranks):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + pos + 1)
    return sorted(scores, key=scores.get, reverse=True)

class MultiQueryRetriever:
    def __init__(self, chunks, embeddings):
        self.chunks, self.embs = chunks, embeddings

    def expand_query(self, query, n=3):
        resp = client.models.generate_content(model="gemini-2.5-flash",
            contents=f"Give {n} alternative phrasings of this question, one per line, questions only.\n\n{query}")
        alts = [l.strip().lstrip("0123456789.-) ") for l in resp.text.strip().split("\n") if l.strip()]
        return [query] + alts[:n]

    def rank_for(self, q):
        qe = embed(q, "RETRIEVAL_QUERY")
        return [i for i, _ in sorted(enumerate(self.embs), key=lambda ie: -cosine(qe, ie[1]))]

    def retrieve(self, query, k=2):
        queries = self.expand_query(query)
        fused = rrf([self.rank_for(q) for q in queries])  # RRF across every phrasing
        return [self.chunks[i] for i in fused[:k]], queries

chunks = [
    "Refund policy: full refund within 7 days. 50 percent from 8-30 days.",
    "GenAI course: 14999 rupees, lifetime access, Discord, GCP credits.",
    "Live classes daily 7 PM IST. Recordings in 2 hours.",
    "Prerequisites: basic Python and high school math.",
]
embs = [embed(c, "RETRIEVAL_DOCUMENT") for c in chunks]
mq = MultiQueryRetriever(chunks, embs)

results, expanded = mq.retrieve("what do I need to know before starting?")
print("Expanded to:", len(expanded), "phrasings")
for c in results: print(f"  {c[:56]}")

# Output:
#   Expanded to: 4 phrasings
#   Prerequisites: basic Python and high school math.
#   Live classes daily 7 PM IST. Recordings in 2 hours.

#### 💡 What just happened

⚡ What just happened?“What do I need to know before starting?” was expanded into 4 phrasings (the original plus 3 alternatives). Each was searched independently, producing a ranked list, and the lists were merged with **Reciprocal Rank Fusion** - rewarding documents ranked high by ANY phrasing, using rank position (1/(60+rank)), never raw scores. **Multi-query catches documents that match ANY phrasing, not just the original. Especially useful when the user’s query is vague or ambiguous.**

## Go Deeper: Rerankers, Architectures, and Evaluation

Steps 5-10 survey the wider field: the reranker landscape, the full query-transformation toolbox, self-correcting RAG architectures, Anthropic's contextual retrieval, the evaluation discipline that gates all of it, and production/India concerns.

---

## 🎯 Concept 5: Reranking Deep Dive - ColBERT, BGE, FlashRank, Jina, Cohere

### Reranking Deep Dive - ColBERT, BGE, FlashRank, Jina, Cohere

Six rerankers compared. Late interaction. Two-stage vs three-stage pipelines.

**Why a whole step on rerankers:** step 2 built one; production teams choose among a dozen, trading accuracy, latency, cost, language coverage, and self-hosting. This step maps the landscape - classic cross-encoders ([BGE](https://github.com/FlagOpen/FlagEmbedding), Cohere, Jina), the FlashRank speed tier, and ColBERT's "late interaction" middle ground - so you pick by requirement, not brand.

> ⚖️ **Analogy**
>
> **Choosing a reranker is choosing a judging panel for a competition.** A single expert judge who scores each finalist thoroughly (a cross-encoder) is most accurate but slow. A panel that pre-scores everyone on a rubric and only deliberates on close calls (ColBERT's late interaction - precompute token-level representations, do the fine comparison cheaply at query time) is faster with most of the accuracy. A quick crowd-vote (FlashRank) is fastest and roughest. You match the panel to the stakes and the clock.
> **Where the analogy breaks down:** judges improve with experience; a frozen reranker does not, and - crucially - even the best panel can only rank the finalists who made it to the stage. The retriever still sets the ceiling, so a better reranker is worth less than better retrieval when recall is the bottleneck.

**📝 `05_reranker_tiers.py`** - *language-python*

In [ ]:
# Same call shape across the tiers - swap the model to move the accuracy/latency dial
# pip install sentence-transformers
from sentence_transformers import CrossEncoder

TIERS = {
    "accurate": "BAAI/bge-reranker-v2-m3",     # open, multilingual, self-host
    "fast":     "cross-encoder/ms-marco-MiniLM-L-6-v2",  # lighter, quicker
}

def rerank(model_id, query, docs, top_k=3):
    ce = CrossEncoder(model_id)
    scores = ce.predict([(query, d) for d in docs])
    return sorted(zip(docs, scores), key=lambda x: -x[1])[:top_k]

# ColBERT-style "late interaction" is the middle tier: precompute token-level
# vectors per doc, then do the fine per-token MaxSim comparison cheaply at query time -
# near cross-encoder quality, closer to bi-encoder speed. Cohere/Jina are hosted APIs.
print(rerank(TIERS["accurate"], "prerequisites?", ["Python + math", "costs 14999"])[0][0])

# Output:
#   Python + math

Before step 5: does upgrading from a good reranker to the best reranker fix a low-recall pipeline?

| Reranker | Type | Latency | Cost | Best For |
|---|---|---|---|---|
| FlashRank | Cross-encoder (4MB) | <~60ms/100 | Free | CPU/serverless, ultra-low latency |
| BGE-reranker-v2-m3 | Cross-encoder (568M) | ~8ms/pair | Free | Self-hosted, multilingual (18+ langs) |
| Jina Reranker v2 | Cross-encoder (278M) | 15× faster BGE | $0.02/M tok | Agentic RAG, code search |
| Cohere Rerank 3.5 | Proprietary | ~392ms | $2/1K search | Enterprise, managed, 100+ langs |
| ColBERTv2 | Late interaction | ~57ms pipeline | Free | First-stage retrieval, explainability |
| ColPali | Visual late interaction | Fast index | Free | Multimodal/visual document retrieval |

> ✅ **Info**
>
> 📈 Reranking impact benchmarks
> **+15-40% retrieval accuracy** across all benchmarks. Hit@1: ~63% → ~83%. Recall@10: ~74% → ~89% (Databricks). MRR: +29-34%. Three-stage pipeline (BM25 + dense + rerank): **+48% total improvement**. Sweet spot: rerank top 50-75 candidates. Beyond 100 shows diminishing returns. ColBERTv2's MaxSim scoring: near cross-encoder accuracy at bi-encoder speed (~57.7ms total). ColPali bypasses OCR entirely for visual documents (ICLR 2025).

#### 💡 What just happened

What just happened? Reranking is the **highest-impact single addition** after hybrid search. BGE-reranker-v2-m3 is the production default for self-hosted (free, multilingual, ~8ms/pair). FlashRank for serverless/edge (4MB, CPU-only). Cohere for managed enterprise. ColBERTv2 occupies a unique middle ground - late interaction gives cross-encoder quality at bi-encoder speed. The 2025 frontier: training rerankers with RL to optimize for downstream LLM generation quality, not just NDCG.

---

## 🎯 Concept 6: Query Transformation - Step-Back, Decomposition, Routing, Expansion

### Query Transformation - Step-Back, Decomposition, Routing, Expansion

Six strategies. When each wins. Query routing = highest single ROI.

**Matching the transform to the query shape:** HyDE and multi-query were two query transforms; this step completes the toolbox. Step-back prompting abstracts a narrow question to a broader one ("what is the 2019 tax rate for X?" becomes "how are tax rates structured?") so retrieval finds the governing document; decomposition splits a multi-hop question into sub-questions; routing decides whether to retrieve at all. The skill is diagnosing which transform a query needs.

> 🧰 **Analogy**
>
> **Query transformation is a librarian rephrasing your request before walking the stacks.** Ask for something too specific ("the 2019 amendment to clause 4b") and a good librarian steps BACK to the governing section first (step-back). Ask a compound question ("compare the refund and cancellation policies") and they SPLIT it into two lookups (decomposition). Ask something they can answer from memory and they do not walk the stacks at all (routing). Same request, different handling, chosen by shape.
> **Where the analogy breaks down:** a librarian rephrases for free from experience; every LLM-driven transform costs a call and latency, and the wrong transform (decomposing a simple query, step-back on a lookup) actively hurts. Routing - deciding WHICH transform, or none - is the highest-ROI move precisely because it prevents paying for transforms that do not fit.

**📝 `06_query_transforms.py`** - *language-python*

In [ ]:
# The transforms are just prompts - the skill is choosing which one
STEP_BACK = "Rewrite this narrow question as the broader question whose answer contains it.\n{q}"
DECOMPOSE = "Break this into the minimal set of standalone sub-questions, one per line.\n{q}"
ROUTE     = "Answer with one word - RETRIEVE, WEB, or DIRECT - for how to handle:\n{q}"

# Examples of what each returns:
#   step-back:  "2019 amendment to clause 4b?"  ->  "How is clause 4 structured?"
#   decompose:  "compare refund vs cancellation" ->  ["refund policy?", "cancellation policy?"]
#   route:      "what is 2 + 2?"                 ->  DIRECT   (no retrieval needed)

def transform(client, template, q):
    return client.models.generate_content(model="gemini-2.5-flash",
        contents=template.format(q=q)).text.strip()

# Output: (transform("...", STEP_BACK, "2019 amendment to clause 4b?"))
#   How is clause 4 structured, and what amendments has it had?

A query is "the 2019 amendment to clause 4b". Which query transform most likely helps?

> 📦 **Info**
>
> 🔎 Query transformation strategies
> - **Query routing (highest ROI):** Classify query → route to appropriate strategy. Simple classifier running in <~1ms improved accuracy ~58% → ~83% (+25%). Three types: LLM-based (flexible), semantic (fast), zero-shot classification. RouteRAG: +10-25% accuracy with meta-cache reducing latency ~150ms → ~30ms.
> - **Step-back prompting (Google DeepMind):** Abstract the query for broader retrieval. "iPhone 13 Pro refresh rate?" → "iPhone 13 Pro technical specs?" roughly +27% on TimeQA and a similar error-rate drop (reported). Best for overly specific queries with dates, IDs, model numbers.
> - **Query decomposition:** Break complex queries into sub-queries. Sequential (multi-hop, each feeds next) or parallel (comparative). DecomposeRAG: large reported gains that grow with hop count (roughly +31% on 2-hop up to much higher on 4-hop). Cost: 2-5x latency.
> - **Query expansion:** LLM generates additional terms, synonyms, hypothetical answers. Marginal gains when hybrid search already in use. Best combined with HyDE.
> - **Sub-question generation:** CoopRAG (NeurIPS 2025): masked uncertain positions. PAR2-RAG: roughly +23.5% over IRCoT on multi-hop (reported) by separating retrieval from reasoning.

#### 💡 What just happened

What just happened?**Query routing is the single highest-ROI improvement** in the entire retrieval stack - a simple classifier can improve accuracy by ~18-25% with near-zero latency cost. The query complexity taxonomy: Class I (simple, explicit) → direct retrieval. Class II (explicit, multi-source) → decomposition. Class III (implicit, single) → step-back/HyDE. Class IV (implicit, multi-source) → full pipeline. Start with routing, add complexity only where benchmarks justify it.

---

## 🎯 Concept 7: Advanced RAG Architectures - Self-RAG, CRAG, Adaptive, Agentic

### Advanced RAG Architectures - Self-RAG, CRAG, Adaptive, Agentic

Reflection tokens. Retrieval correction. Dynamic routing. ReAct loops.

**When retrieval becomes a loop:** everything so far runs retrieval once. These architectures let the system CHECK its own retrieval and act on the result - retrieve again, correct with web search, or route by difficulty. Self-RAG bakes the checks into the model (reflection tokens, needs training); CRAG wraps any frozen LLM with a lightweight grader and three fallbacks; Adaptive routes by question difficulty.

> 🏛️ **Analogy**
>
> **These architectures are a researcher who checks their sources before writing.** A naive RAG grabs the first books off the shelf and writes. A careful researcher reads what they pulled, asks "is this actually on-topic and sufficient?", and if not, goes back to the stacks or switches to the web (CRAG's correct / ambiguous / incorrect paths). A seasoned one first judges how hard the question is and picks their method accordingly (Adaptive) - a quick fact needs no library trip at all.
> **Where the analogy breaks down:** a human researcher's self-check is free and intuitive; each machine reflection step is another model call and more latency, and Self-RAG's "intuition" has to be TRAINED into the model in the first place. The loops buy accuracy on hard, multi-hop questions - and waste money on easy ones, which is why routing gates them.

**📝 `07_crag_route.py`** - *language-python*

In [ ]:
# CRAG in one function: grade the retrieved docs, then take one of three paths
def crag_step(grade, docs, web_search):
    if grade == "correct":      return docs                          # use as-is
    if grade == "ambiguous":    return docs + web_search()            # augment
    return web_search()                                            # "incorrect": discard, fall back

# A lightweight grader (small model or classifier) scores retrieval quality;
# Self-RAG instead BAKES this decision into the model via reflection tokens
# (needs fine-tuning); Adaptive-RAG routes by predicted question difficulty.
print(crag_step("incorrect", [], lambda: ["fresh web result"]))

# Output:
#   ['fresh web result']

CRAG grades its retrieved documents as "incorrect". What does it do next?

| Architecture | Key Mechanism | Best For | Plug-and-Play? |
|---|---|---|---|
| Self-RAG | 4 reflection tokens (Retrieve, ISREL, ISSUP, ISUSE) | Max factual accuracy (~55.8% vs ~14.7%) | ❌ Needs fine-tuning |
| CRAG | Retrieval evaluator + 3-action routing | Best effort/improvement ratio | ✅ Plug-and-play |
| Adaptive RAG | Complexity classifier → strategy routing | Mixed-complexity queries, latency savings | Partially (classifier) |
| Agentic RAG | ReAct loops + tools (LangGraph) | Multi-tool workflows | ✅ LangGraph |
| FLARE | Confidence-based iterative retrieval | Long-form generation | ✅ |
| IRCoT | Interleaved CoT + retrieval | Multi-hop reasoning (+21 pts) | ✅ |
| GraphRAG | KG + community summaries | Global/relational queries (~70-80% win) | Partially |

#### 💡 What just happened

What just happened? The RAG architecture landscape has fragmented into **seven distinct patterns**, each solving a different failure mode. CRAG is the best starting point (plug-and-play, +26.7 points). Adaptive RAG saves ~25-35% latency by skipping retrieval for simple queries. Agentic RAG via LangGraph gives maximum flexibility. GraphRAG excels at "what are the main themes across this dataset?" questions where standard RAG fails completely. **Cap agentic loops at 3 retrieval cycles** to avoid retrieval thrash.

---

## 🎯 Concept 8: Contextual Retrieval (Anthropic) - Stamp Chunks Before Indexing

### Contextual Retrieval (Anthropic) - Stamp Chunks Before Indexing

LLM-generated context per chunk. Prompt template. Cost analysis. Stacking benchmarks.

**Fixing the problem upstream of retrieval:** everything so far improved how you SEARCH; contextual retrieval improves what you INDEX. A chunk torn from a document loses its context - "the fee is 2 percent" no longer says which fee - so neither dense nor BM25 can match it. Anthropic's fix: an LLM prepends one situating sentence to each chunk before embedding AND before [BM25](https://github.com/xhluca/bm25s) indexing, so the chunk carries its own context. Stacked with hybrid and reranking, it drives failures down sharply (Anthropic-reported).

> 🔖 **Analogy**
>
> **Contextual retrieval is stamping every photocopied page before it goes in the filing cabinet.** Tear a page from a report and file it loose, and "the fee is 2 percent" is unfindable - which report? which section? A good clerk stamps each copy first: "Netsetos 2026 refund policy, cancellation fees, page 4." Now the same page is findable by topic, by document, by date - and it helps EVERY way someone might search for it later (both the meaning index and the keyword index), because the stamp is part of the text.
> **Where the analogy breaks down:** a rubber stamp is free; here the "stamp" is an LLM call per chunk. It only pays off because it is a ONE-TIME ingestion cost (not per query) and prompt caching makes re-reading the parent document cheap - so it is affordable at corpus scale, unlike running an LLM on every search.

An orphan chunk reads only "The fee is 2 percent." Why does contextual retrieval help find it?

- Scene 1: the orphan chunk - "the fee is 2 percent" torn from its document; a query "cancellation fee?" cannot match it and a reranker has nothing to rescue.

- Scene 2: stamp it - the LLM reads the whole document + the chunk and prepends ONE situating sentence naming the fee and the document.

- Scene 3: two indexes, both improved - the stamped chunk is embedded AND added to BM25; now the same query matches at rank 1 in BOTH.

- Scene 4: the stack - naive dense, then +hybrid, then +reranking, then +contextual; the retrieval-failure bar drops at each layer (Anthropic-reported), and prompt caching keeps the one-time stamping cheap.

Controls: Play/Pause, Reset, speed. Contextual retrieval fixes chunk-context-loss at ingestion.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

> 📦 **Info**
>
> 🤖 Anthropic's contextual retrieval - implementation
> Prepend LLM-generated context (50-100 tokens) to each chunk before embedding AND BM25 indexing. "Revenue grew ~3%" becomes "This chunk is from ACME Q2 2023 SEC filing; previous quarter revenue was $314M. Revenue grew ~3%."
> - **Contextual Embeddings only:** -35% retrieval failures
> - **+ Contextual BM25:** -49% failures (the key combination)
> - **+ Reranking on top:** Anthropic reports about -67% retrieval failures overall
> - **Cost:** Anthropic reports roughly $1 per million document tokens (one-time at index time) with prompt caching
> - **Skip when:** Knowledge base <200K tokens (~500 pages) - just stuff everything in context
> - **Use when:** Chunks lose meaning when separated (SEC filings, technical docs, codebases)

#### 💡 What just happened

What just happened? Contextual retrieval is the **highest-ROI pre-processing technique**: one-time indexing cost for permanent retrieval improvement. All benefits stack additively: contextual embeddings, then contextual BM25, then reranking - Anthropic reports roughly -35%, -49%, and -67% retrieval-failure reductions as they stack. Use Claude Haiku for cost efficiency. Prompt caching is essential - loads entire document once, references for each chunk, reducing cost by up to ~90%.

---

## 🎯 Concept 9: Retrieval Evaluation - NDCG, MRR, RAGAS, DeepEval, Golden Test Sets

### Retrieval Evaluation - NDCG, MRR, RAGAS, DeepEval, Golden Test Sets

Five metrics. Two frameworks. CI/CD integration. Silver → gold pipeline.

**The step that makes all the others honest:** every layer in this lesson claims an uplift - but on YOUR corpus, some help and some just add latency. Evaluation is how you tell. The key distinction: order-aware metrics (MRR, NDCG) reward putting the right document HIGH, while order-unaware metrics (recall@k, precision@k) only ask if it is in the top k - so "answer at rank 1" and "answer at rank 10" score identically on recall@10. RAGAS and DeepEval automate context precision/recall/faithfulness on a golden test set.

> 📈 **Analogy**
>
> **Retrieval metrics are the difference between "did the team make the playoffs" and "what seed did they get."** Recall@10 is the playoff question - in or out of the top 10, a yes/no that ignores position. MRR and NDCG are the SEED - they reward finishing first, not just qualifying, because in retrieval a fact at rank 1 gets read and a fact at rank 10 often does not. RAGAS is the season-long analytics package that scores not just placement but whether the retrieved context actually supports the answer.
> **Where the analogy breaks down:** a sports seed is objective; retrieval metrics need a GOLDEN SET - human-labeled "right answers" for each query - and building that set is the real work. Automated RAGAS scores are only as trustworthy as the questions and labels you feed them, which is why the step ends on golden-test-set discipline, not the metrics themselves.

**📝 `09_metrics_by_hand.py`** - *language-python*

In [ ]:
# NDCG and MRR worked by hand on one query - see why order matters
import math

# The one relevant doc is at rank 3 (0-indexed position 2) in our top-5
relevance = [0, 0, 1, 0, 0]   # 1 = the right answer

# MRR: reciprocal of the FIRST relevant rank -> 1/3
first = next(i for i, r in enumerate(relevance) if r) + 1
mrr = 1 / first

# NDCG@5: DCG = sum(rel / log2(rank+1)); normalize by the ideal (answer at rank 1)
dcg   = sum(r / math.log2(i + 2) for i, r in enumerate(relevance))
ideal = 1 / math.log2(1 + 1)   # the one relevant doc, placed at rank 1
print(f"MRR = {mrr:.2f}   NDCG@5 = {dcg/ideal:.2f}")

# recall@5 would be 1.0 here (the answer IS in the top 5) - it cannot tell rank 3
# from rank 1. MRR and NDCG can, which is why they are the order-aware metrics.

# Output:
#   MRR = 0.33   NDCG@5 = 0.50

Doc A holds the answer at rank 1; doc set B holds it at rank 9. Which metric scores them the same?

> ✅ **Info**
>
> 📊 Retrieval metrics decision framework
> - **MRR:** How quickly first correct answer appears. Target >0.8. Use for single-answer QA.
> - **MAP:** Average precision at each relevant doc's rank. Use when multiple relevant docs exist.
> - **NDCG@K:** Discounted cumulative gain, normalized. Default on MTEB. Use for graded relevance. Target >0.8.
> - **Recall@K:** Fraction of relevant docs found. Use when completeness matters (legal, compliance).
> - **RAGAS:** Faithfulness (claims supported by context), answer relevancy, context precision, context recall. Reference-free. Target: faithfulness >0.8, precision >0.8.
> - **DeepEval:** pytest-style LLM evaluation with 50+ metrics. CI/CD integration: `deepeval test run test_rag.py`. Start with 0.65 thresholds, tighten as baseline improves.

#### 💡 What just happened

What just happened?**Evaluation infrastructure is non-negotiable for production RAG.** Without automated quality gates, improvements in one area silently degrade another. The silver → gold pipeline: synthetic test data (RAGAS TestsetGenerator) → SME review → 50-200 golden QA pairs + 300 synthetic. Four-layer stack: offline golden tests → CI/CD DeepEval gates → A/B testing → continuous monitoring. Key insight: basic RAG is ~65% accurate at ~92% confidence (dangerously overconfident). Advanced RAG with proper evaluation: ~92% accurate at ~88% confidence (properly calibrated).

---

## 🎯 Concept 10: Production Optimization + India Retrieval - Latency, Caching, Cross-Lingual

### Production Optimization + India Retrieval - Latency, Caching, Cross-Lingual

Latency budgets. Semantic caching. Dual retrieval for Hindi. ChrF evaluation.

**Making the stack shippable:** a four-layer retrieval stack can blow a latency budget and a bill. This step is the production reckoning - where the time and money go, what to cache, and the India-specific reality that a Hindi question against English documents falls into a cross-lingual gap no single retriever bridges.

> ⏱️ **Analogy**
>
> **Production retrieval is a pit stop: every added layer is another second on the clock.** Hybrid, HyDE, multi-query, and reranking each buy accuracy but spend milliseconds; a race engineer budgets the stop, caches what repeats (semantic caching answers near-identical queries for free), and cuts any step that does not earn its time. For India, the pit crew runs TWO lanes at once - dense retrieval on the Hindi query and BM25 on an English translation, fused with RRF - because each lane catches what the other drops.
> **Where the analogy breaks down:** a pit stop has one clear clock; retrieval trades against three axes at once - latency, cost, and accuracy - and the right cut depends on which one is binding. And ChrF (not BLEU) is the correct quality gauge for Indic output, because a single metric tuned for English silently mis-scores rich-morphology languages.

**📝 `10_semantic_cache.py`** - *language-python*

In [ ]:
# Semantic cache: answer a near-identical query for free by matching its embedding
import numpy as np

def cache_lookup(q_emb, cache, threshold=0.95):
    """cache: list of (embedding, answer). Return a hit if close enough."""
    for emb, answer in cache:
        sim = float(np.dot(q_emb, emb) / (np.linalg.norm(q_emb) * np.linalg.norm(emb)))
        if sim >= threshold:
            return answer   # exact-enough match - skip retrieval + generation entirely
    return None

# "what is the refund window?" and "how many days for a refund?" embed close,
# so the second query is served from cache - a large share of production traffic repeats.

q_emb = np.array([0.1, 0.2, 0.3])   # any query vector; here it matches the cached one
print(cache_lookup(q_emb, [(q_emb, "Refunds within 7 days.")]))

# Output:
#   Refunds within 7 days.

A Hindi question must retrieve from English documents, and each retrieval layer adds latency. First move?

> 📦 **Info**
>
> 🇮🇳 Production + India-specific patterns
> - **Latency budget:** Query embed ~10-30ms → Retrieval ~30-80ms → Reranking ~50-150ms → Generation ~300-800ms. Query rewriting adds 2.4× TTFT - most expensive pre-generation step.
> - **Semantic caching (GPTCache):** Cache results for similar queries (cosine >0.85). Realistic savings: ~67% API call reduction (not ~80-95%). Security: SAFE-CACHE reduces adversarial attacks from ~53% to ~14%.
> - **Cross-lingual Hindi-English:** Dual retrieval: Branch 1 = MuRIL/multilingual-e5 dense on Hindi corpus. Branch 2 = translate query to English + BM25 on English corpus. Union via RRF → rerank with BGE-reranker-v2-m3 (supports Hindi). Cross-lingual accuracy: ~60-80% of monolingual.
> - **Sarvam AI:** Token fertility 1.4-2.1 (vs 4-8 standard) = 2-4× cost savings. Sarvam-M supports formal + code-mixed + Romanized scripts.
> - **Hinglish queries:** Token-level language ID (HingBERT, F1=~98.8%) → transliterate Romanized Hindi (IndicXlit) → embed both normalized + original → union candidates.
> - **ChrF evaluation:** Character n-gram F-score outperforms BLEU for morphologically rich Indic languages. Use alongside BLEU, not alone. IndicCOMET for highest human correlation.

#### 💡 What just happened

What just happened? Production retrieval is a **latency engineering problem**. The typical 1.2s budget is dominated by generation (~300-800ms) - optimize retrieval and reranking to stay under ~200ms combined. Semantic caching saves ~67% of API calls. For India: dual retrieval (Hindi dense + English BM25 + RRF) is the proven pattern. Sarvam AI's token efficiency cuts embedding costs 2-4×. ChrF is the correct evaluation metric for Indic languages - BLEU penalizes inflectional variations unfairly.

> 📦 **Info**
>
> 🔗 Where this goes next (Module 8)
> - **In Lesson 8.5 (RAG with LangChain)** and **8.6 (LlamaIndex)**, this whole stack - hybrid retrievers, rerankers, query transforms - becomes composable behind one interface; we will come back to the retrieve-then-rerank pattern as a few lines of framework code.
> - **In Lesson 8.10 (Agentic RAG)**, step 7's Self-RAG / CRAG loops grow into full agent architectures with tool use and multi-hop retrieval.
> - **In Lesson 8.11 (RAG Evaluation & RAGAS)**, step 9's NDCG/MRR and RAGAS metrics become a full evaluation harness wired into CI - the discipline that tells you which layer here actually earned its latency.

## Stacking Techniques - The Production Pipeline

None of these techniques competes with the others - they STACK. The production order, cheapest-signal-first:

> 📦 **Info**
>
> Recommended stack (in order)
> - **Multi-Query:** expand to about 3 variants (a modest recall gain).
> - **Hybrid Search:** BM25 + dense per variant, so exact tokens and meaning both count.
> - **Merge + Deduplicate:** RRF (k=60) across all results.
> - **Re-Rank:** cross-encoder on the top ~10, keep the top 3 (the biggest precision jump).
> Each layer is independently measurable (step 9), and the combined lift over naive dense RAG is large - but every layer adds latency and cost, so add them in this order and stop when your eval (step 9) says you have enough.

## Key Takeaways

> 📦 **Info**
>
> - **Hybrid search is the non-negotiable default.** Dense catches meaning, BM25 catches exact tokens; RRF (k=60) fuses their complementary blind spots - the biggest single jump over naive dense retrieval.
> - **Retrieve wide, then rerank narrow.** A fast bi-encoder shortlists top-100; a slow cross-encoder reads query and document jointly to pick the precise top-5.
> - **The retriever sets the ceiling.** A reranker only reorders what retrieval found - fix retrieval (hybrid, contextual) before buying a fancier reranker.
> - **Query transforms trade an LLM call for recall.** HyDE (embed a hypothetical answer), multi-query (several phrasings + RRF), and step-back help vague or hard queries - measure before adding.
> - **Contextual retrieval fixes chunk-context-loss at ingestion,** and evaluation (NDCG/MRR order-aware, RAGAS context precision/recall) is what tells you which layer to keep.

*Practice exercises are in the companion practice notebook.*

Eight exercises, easy to hard. Each builds or measures one layer of the retrieval stack.

---

## 🎓 Summary

You've completed the practical part of **Advanced Retrieval- Beyond Naive Cosine**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-8_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-8.4.html` - regenerate this notebook whenever the source HTML is updated.*
